# Tally held-out test (final exam)

Run the **base** Tiny Aya model and the **v7** LoRA adapter once on the sealed held-out project descriptions (5172 Akwaaba, 5432 Kwahu, 4329 CRCF). Scores are written to `heldout_comparison.json`.

The held-out gold answer key is **not in this repository**. Attach it at runtime (Kaggle dataset or `TALLY_HELDOUT_DIR`) together with `heldout_messages.jsonl`, `heldout_meta.jsonl`, `tally_merge.py`, and `score_extractions.py`. Project 2928 (North Bandai) is not scored: no local document or key was collected.

**Setup**

1. Settings → Accelerator → GPU T4 x1, Internet on.
2. Attach the private held-out data dataset and, if used, `renhuang01/tally-hf-token`.
3. Add-ons → Secrets → attach `HF_TOKEN`. The token is never printed.
4. Attach the v7 LoRA adapter (or a zip containing `adapter_config.json`).
5. Save & Run All. To skip a already-scored base run, set the environment variable `TALLY_SKIP_BASE=1`.


In [ ]:
# === 1. SETUP: install packages, log in, and set paths ===

import os
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def _work_root():
    if Path("/kaggle/working").is_dir():
        return Path("/kaggle/working")
    env = os.environ.get("TALLY_ROOT")
    p = Path(env).expanduser() / "sft_out" / "heldout_local" if env else (Path.cwd() / "tally_runs" / "heldout")
    p.mkdir(parents=True, exist_ok=True)
    return p

WORK_ROOT = _work_root()
HF_CACHE_DIR = Path("/kaggle/tmp/hf") if Path("/kaggle/working").is_dir() else (WORK_ROOT / "hf_cache")
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE_DIR)
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE_DIR)

# Do not reinstall torch. transformers>=5 matches the extraction notebook.
%pip install -q bitsandbytes peft "transformers>=5"
# PEFT rejected preinstalled torchao 0.10 on the T4 image used for v7.
%pip uninstall -y -q torchao

from huggingface_hub import login


def get_hf_token():
    # 1) the notebook's HF_TOKEN Kaggle secret; 2) a private attached dataset
    # file hf_token.txt. The token is never printed.
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN")
        if tok and tok.strip():
            return tok.strip(), "Kaggle secret"
    except Exception as exc:
        print("HF_TOKEN secret not available:", type(exc).__name__)
    search_roots = [Path("/kaggle/input")]
    if os.environ.get("TALLY_ROOT"):
        search_roots.append(Path(os.environ["TALLY_ROOT"]).expanduser())
    search_roots.append(Path.cwd())
    for root in search_roots:
        if not root.exists():
            continue
        for f in sorted(root.rglob("hf_token.txt")):
            tok = f.read_text(encoding="utf-8").strip()
            if tok:
                return tok, "private dataset file"
    raise RuntimeError("No Hugging Face token: attach the HF_TOKEN secret or a private hf_token.txt file.")


_hf_tok, _hf_src = get_hf_token()
login(_hf_tok)
os.environ["HF_TOKEN"] = _hf_tok
print("Logged in to Hugging Face via", _hf_src)
del _hf_tok

import torch

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}  "
              f"{torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB")
else:
    print("WARNING: no GPU visible. Turn on Settings → Accelerator → GPU T4 x1 and restart.")

print("cuda:", torch.cuda.is_available(), "torch", torch.__version__)
import importlib.metadata as _md
for _p in ("transformers", "trl", "peft", "datasets", "bitsandbytes", "accelerate"):
    try:
        print(f"{_p}=={_md.version(_p)}")
    except Exception:
        print(_p, "not installed")


In [ ]:
# === 2. PATHS: locate held-out chunks, the gold key (runtime only), and the v7 adapter ===

import json, sys, time, traceback, gc
from pathlib import Path
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM

MID = "CohereLabs/tiny-aya-global"
HF_CACHE = str(HF_CACHE_DIR)
RESULTS_DIR = WORK_ROOT / "heldout"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def _search_roots():
    roots = []
    for env_key in ("TALLY_HELDOUT_DIR", "TALLY_ROOT"):
        if os.environ.get(env_key):
            roots.append(Path(os.environ[env_key]).expanduser())
    for p in (Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()):
        roots.append(p)
    return roots

def find_heldout_dir():
    for root in _search_roots():
        if not root.exists():
            continue
        hits = sorted(root.rglob("heldout_messages.jsonl"))
        if hits:
            return hits[0].parent
    raise FileNotFoundError(
        "heldout_messages.jsonl not found. Attach the held-out dataset or set TALLY_HELDOUT_DIR."
    )

DATA_DIR = find_heldout_dir()

def find_adapter_dir():
    roots = list(_search_roots())
    roots.append(DATA_DIR)
    for root in roots:
        if not root.exists():
            continue
        hits = sorted(p for p in root.rglob("adapter_config.json") if "checkpoint-" not in str(p))
        if hits:
            return hits[0].parent
        for z in sorted(root.rglob("tally-lora*.zip")):
            import zipfile
            dest = WORK_ROOT / "tally-lora-unz"
            zipfile.ZipFile(z).extractall(dest)
            hits = sorted(dest.rglob("adapter_config.json"))
            if hits:
                return hits[0].parent
    return DATA_DIR / "tally-lora"

ADAPTER_DIR = find_adapter_dir()
sys.path.insert(0, str(DATA_DIR))
# Also allow importing the checked-in scorer from a clone of this repo.
_src = Path.cwd() / "src"
if _src.is_dir():
    sys.path.insert(0, str(_src))

def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

score_rows = load_jsonl(DATA_DIR / "heldout_messages.jsonl")
score_meta = load_jsonl(DATA_DIR / "heldout_meta.jsonl")
_key_path = DATA_DIR / "answer_key_heldout.json"
if not _key_path.is_file():
    raise FileNotFoundError(
        "answer_key_heldout.json is required at runtime and is intentionally not in the git repository."
    )
answer_key = json.loads(_key_path.read_text(encoding="utf-8"))
from tally_merge import FIELDS, merge_notes
from score_extractions import score_documents, format_summary_table
print("DATA_DIR", DATA_DIR, "| chunks", len(score_rows), "| docs", sorted(answer_key), "| adapter", ADAPTER_DIR.exists())

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
# === 6. HELPERS for generation and scoring ===

import re
from collections import defaultdict

from peft import PeftModel


def parse_json(text):
    # Parse the FIRST complete JSON object. (v4: a reply that repeats the
    # object, e.g. '{...}{...}', used to fail with "Extra data".) Same parser
    # for base and fine-tuned, so the comparison stays fair.
    start = text.find("{")
    if start == -1:
        raise json.JSONDecodeError("no JSON object found", text, 0)
    try:
        obj, _ = json.JSONDecoder().raw_decode(text[start:])
        return obj
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.S)
        return json.loads(match.group(0))


def generate_reply(model, tok, prompt, max_new_tokens=900):
    messages = [{"role": "user", "content": prompt}]
    inputs = tok.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            # stop at <|END_RESPONSE|> or <|END_OF_TURN_TOKEN|>
            eos_token_id=[i for i in (tok.convert_tokens_to_ids("<|END_RESPONSE|>"), tok.eos_token_id) if i is not None],
            pad_token_id=tok.pad_token_id,
        )
    reply_ids = output[0][inputs["input_ids"].shape[1]:]
    return tok.decode(reply_ids, skip_special_tokens=True)


def extract_chunk_text(user_content):
    # Strip the shared extraction prompt, leaving the page-marked chunk.
    marker = "\nPages "
    pos = user_content.find(marker)
    if pos == -1:
        return user_content
    rest = user_content[pos + 1:]  # starts with 'Pages ...'
    # Skip the 'Pages N–M:' header line.
    nl = rest.find("\n")
    return rest[nl + 1:].lstrip() if nl != -1 else rest


def run_val_extraction(model, tok, label):
    # Run every held-out chunk, merge per document, and score against the key.
    if not score_rows or not score_meta:
        raise RuntimeError("held-out score set is empty or missing")
    per_doc_notes = defaultdict(list)
    per_doc_text = defaultdict(list)
    raw_rows = []
    for i, (row, meta) in enumerate(zip(score_rows, score_meta)):
        doc = meta["doc"]
        prompt = row["messages"][0]["content"]
        print(f"[{label}] {i + 1}/{len(score_rows)}  {doc} chunk {meta.get('chunk_index')}")
        try:
            reply = generate_reply(model, tok, prompt)
            parsed = parse_json(reply)
            err = None
        except Exception as exc:
            reply = locals().get("reply", "")
            parsed = {"_error": f"{type(exc).__name__}: {exc}", "raw": reply[:2000]}
            err = str(exc)
            print("  parse/generate error:", err)
        per_doc_notes[doc].append(parsed)
        per_doc_text[doc].append(extract_chunk_text(prompt))
        raw_rows.append({
            "doc": doc,
            "chunk_index": meta.get("chunk_index"),
            "reply": reply,
            "parsed": parsed,
            "error": err,
        })
        if torch.cuda.is_available() and (i + 1) % 4 == 0:
            torch.cuda.empty_cache()

    merged_by_doc = {}
    for doc, notes in per_doc_notes.items():
        doc_text = "\n\n".join(per_doc_text[doc])
        merged_by_doc[doc] = merge_notes(notes, doc_text=doc_text)

    scores = score_documents(merged_by_doc, answer_key, docs=sorted(merged_by_doc))
    print()
    print(format_summary_table(scores, f"{label} — HELD-OUT"))
    out = {
        "label": label,
        "merged": merged_by_doc,
        "scores": scores,
        "raw": raw_rows,
    }
    path = RESULTS_DIR / f"{label}_heldout_scores.json"
    path.write_text(json.dumps(out, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    print("wrote", path)
    return out


def load_base_model():
    tok = AutoTokenizer.from_pretrained(MID, cache_dir=HF_CACHE)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    # Inference in float16, matching the extraction notebook, unless memory is tight.
    try:
        model = AutoModelForCausalLM.from_pretrained(
            MID,
            dtype=torch.float16,
            device_map={"": 0},
            cache_dir=HF_CACHE,
        )
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(
            MID,
            torch_dtype=torch.float16,
            device_map={"": 0},
            cache_dir=HF_CACHE,
        )
    model.eval()
    return tok, model


def release(*objs):
    for obj in objs:
        try:
            del obj
        except Exception:
            pass
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
# === 3. RUN base, then v7 ===
# Set TALLY_SKIP_BASE=1 to reuse a base_heldout_scores.json already in RESULTS_DIR.

SKIP_BASE = os.environ.get("TALLY_SKIP_BASE", "").strip().lower() in {"1", "true", "yes"}
base_results = None
if SKIP_BASE:
    prior = RESULTS_DIR / "base_heldout_scores.json"
    if prior.is_file():
        base_results = json.loads(prior.read_text(encoding="utf-8"))
        print("Loaded existing base scores from", prior)
    else:
        print("TALLY_SKIP_BASE is set but no base_heldout_scores.json was found; will run base.")
        SKIP_BASE = False

if not SKIP_BASE:
    t0 = time.time()
    try:
        base_tok, base_model = load_base_model()
        base_results = run_val_extraction(base_model, base_tok, "base")
    except Exception as exc:
        print("base scoring failed:", type(exc).__name__, exc)
        traceback.print_exc()
    finally:
        base_model = None
        base_tok = None
        free_gpu()
    print("base done in %.1f min" % ((time.time() - t0) / 60))

t0 = time.time()
ft_results = None
try:
    ft_tok, base_for_adapter = load_base_model()
    ft_model = PeftModel.from_pretrained(base_for_adapter, str(ADAPTER_DIR))
    ft_model.eval()
    if hasattr(ft_model, "config"):
        ft_model.config.use_cache = True
    ft_results = run_val_extraction(ft_model, ft_tok, "v7")
except Exception as exc:
    print("v7 scoring failed:", type(exc).__name__, exc)
    traceback.print_exc()
finally:
    ft_model = None
    base_for_adapter = None
    ft_tok = None
    free_gpu()
print("v7 done in %.1f min" % ((time.time() - t0) / 60))


In [ ]:
# === 4. COMPARE and write scores ===
# This cell prints gold values from the runtime key. Do not commit the output file
# if it still contains gold_value fields; the checked-in results/heldout_scores.json
# is a sanitized aggregate.

def overall_line(name, s):
    o = s["overall"]
    return (f"{name:<10} stated={o['stated_accuracy']:.1%} ({o['stated_correct']}/{o['stated_n']})  acc+abs={o['accuracy_incl_abstention']:.1%}  "
            f"correct={o['correct']} wrong={o['wrong']} conflict={o['conflict-unresolved']} abstained={o['correctly-abstained']} hallucinated={o['hallucinated']}")

bs = base_results["scores"] if base_results else None
fs = ft_results["scores"] if ft_results else None
if bs:
    print(overall_line("base", bs))
    print(format_summary_table(bs, "BASE — held-out"))
if fs:
    print(overall_line("v7", fs))
    print(format_summary_table(fs, "V7 — held-out"))
if bs and fs:
    for a, b in zip(bs["pairs"], fs["pairs"]):
        if a["category"] != b["category"]:
            print(f"  {a['doc']} / {a['field']}: {a['category']} -> {b['category']}")

comparison = {
    "heldout_docs": sorted(answer_key),
    "n_chunks": len(score_rows),
    "base": bs,
    "v7": fs,
    "base_merged": base_results["merged"] if base_results else None,
    "v7_merged": ft_results["merged"] if ft_results else None,
}
for p in (RESULTS_DIR / "heldout_comparison.json", WORK_ROOT / "heldout_comparison.json"):
    p.write_text(json.dumps(comparison, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print("wrote heldout_comparison.json (contains runtime gold; keep it out of git)")
